# Week 4 Community Contribution - AdnanGobeljic

This notebook implements a Week 4 style **Code Assistant** with a Gradio UI.

It includes:
- Python-to-target-language conversion (C++, Rust, Go, JavaScript)
- Python docstring and inline comment generation
- Python unit test generation
- Model switching between OpenAI and local Ollama
- Streaming output in the UI

In [ ]:
# imports

import os
import re
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [ ]:
# setup clients and defaults

load_dotenv(override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = None
if openai_api_key and openai_api_key.strip() == openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key)

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def is_ollama_running():
    try:
        response = requests.get("http://localhost:11434", timeout=2)
        return response.status_code == 200
    except Exception:
        return False

OLLAMA_AVAILABLE = is_ollama_running()


In [ ]:
# prompt templates and helpers

SYSTEM_PROMPT = """
You are a senior software engineer and code quality reviewer.
You produce precise, production-focused code outputs.
Return only requested code output, with no markdown fences and no extra commentary.
"""

def strip_code_fences(text):
    cleaned = (text or "").strip()
    cleaned = re.sub(r"^```(?:[a-zA-Z0-9_+-]+)?\\s*", "", cleaned)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    return cleaned


def resolve_client_and_model(model_choice):
    if model_choice == "GPT (OpenAI)":
        if openai_client is None:
            return None, None, "OpenAI is not configured. Add OPENAI_API_KEY to .env."
        return openai_client, OPENAI_MODEL, None

    if model_choice == "Llama (Ollama)":
        if not OLLAMA_AVAILABLE:
            return None, None, "Ollama is not running. Start it with: ollama serve"
        return ollama_client, OLLAMA_MODEL, None

    return None, None, "Unknown model choice."


def build_user_prompt(task, source_code, target_language, test_framework):
    source_code = (source_code or "").strip()

    if task == "Convert Python Code":
        return f"""
Convert the following Python code into optimized {target_language}.

Requirements:
- Preserve functional behavior.
- Prefer performant implementation and clean structure.
- Keep output as valid {target_language} code only.

Python code:
{source_code}
""".strip()

    if task == "Add Docstrings and Comments":
        return f"""
Improve this Python code by adding:
- PEP-257 style docstrings where missing
- concise inline comments for non-obvious logic

Constraints:
- Do not change runtime behavior.
- Do not rename variables or functions unless absolutely required to fix syntax.
- Return complete updated Python code only.

Python code:
{source_code}
""".strip()

    return f"""
Generate {test_framework} tests for the following Python code.

Requirements:
- Cover normal and edge cases.
- Include at least one negative/error-path test where appropriate.
- Return test code only.

Python code:
{source_code}
""".strip()


In [ ]:
def run_code_assistant_stream(source_code, task, model_choice, target_language, test_framework):
    if not (source_code or "").strip():
        yield "Please paste source code first.", ""
        return

    client, model_id, error = resolve_client_and_model(model_choice)
    if error:
        yield error, ""
        return

    user_prompt = build_user_prompt(task, source_code, target_language, test_framework)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    partial = ""
    try:
        stream = client.chat.completions.create(
            model=model_id,
            messages=messages,
            stream=True,
        )

        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            if delta:
                partial += delta
                yield "Generating...", strip_code_fences(partial)

        yield f"Done with {model_choice}.", strip_code_fences(partial)
    except Exception:
        try:
            response = client.chat.completions.create(model=model_id, messages=messages)
            text = strip_code_fences(response.choices[0].message.content or "")
            yield f"Done with {model_choice}.", text
        except Exception as exc:
            yield f"Generation failed: {exc}", ""


In [ ]:
with gr.Blocks(title="Week 4 Code Assistant - AdnanGobeljic") as demo:
    gr.Markdown("# Week 4 - Multi-Model Code Assistant")
    gr.Markdown("Convert code, add docs/comments, or generate tests using GPT or local Llama.")

    source_code = gr.Textbox(
        label="Source code (Python)",
        lines=18,
        placeholder="Paste your Python code here...",
    )

    with gr.Row():
        task = gr.Dropdown(
            choices=[
                "Convert Python Code",
                "Add Docstrings and Comments",
                "Generate Unit Tests",
            ],
            value="Convert Python Code",
            label="Task",
        )
        model_choice = gr.Dropdown(
            choices=["GPT (OpenAI)", "Llama (Ollama)"],
            value="GPT (OpenAI)",
            label="Model",
        )

    with gr.Row():
        target_language = gr.Dropdown(
            choices=["C++", "Rust", "Go", "JavaScript"],
            value="C++",
            label="Target language (for conversion)",
        )
        test_framework = gr.Dropdown(
            choices=["pytest", "unittest"],
            value="pytest",
            label="Test framework (for tests)",
        )

    run_btn = gr.Button("Run", variant="primary")

    status = gr.Textbox(label="Status", interactive=False)
    output_code = gr.Textbox(label="Generated output", lines=18)

    run_btn.click(
        fn=run_code_assistant_stream,
        inputs=[source_code, task, model_choice, target_language, test_framework],
        outputs=[status, output_code],
    )

    gr.Examples(
        examples=[
            [
                "def fib(n):\n    if n <= 1:\n        return n\n    return fib(n-1) + fib(n-2)",
                "Convert Python Code",
                "GPT (OpenAI)",
                "C++",
                "pytest",
            ],
            [
                "def normalize_scores(values):\n    total = sum(values)\n    return [v / total for v in values]",
                "Add Docstrings and Comments",
                "GPT (OpenAI)",
                "C++",
                "pytest",
            ],
            [
                "def is_even(x):\n    return x % 2 == 0",
                "Generate Unit Tests",
                "Llama (Ollama)",
                "C++",
                "pytest",
            ],
        ],
        inputs=[source_code, task, model_choice, target_language, test_framework],
        label="Examples",
    )


In [ ]:
demo.launch()

## Notes

- For OpenAI mode, set `OPENAI_API_KEY` in `.env`.
- For Ollama mode, run `ollama serve` and ensure your model (default `llama3.2`) is pulled.
- This notebook is intentionally lightweight and focused on Week 4 coding workflow patterns.